<a href="https://colab.research.google.com/github/naey008/Profile-Website/blob/main/sesi06_praktikum_minimax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesi 6 · Praktikum: Minimax dan alpha-beta
**Artificial Intelligence · Universitas Cakrawala** · Pertemuan 6

Di Sesi 5, petanya diam: jalan tidak berubah karena langkahmu. Hari ini ada **lawan** yang membalas setiap langkahmu dan ingin kamu kalah. Komputer memilih langkah dengan cara yang sama seperti di lembar kertas tadi: anggap lawan selalu bermain terbaik, hitung nilai dari bawah ke atas, lalu pilih langkah yang nilainya terbaik.

Semua kode di notebook ini ditulis di depan kamu. Nomor aturan di komentar kode sama dengan nomor aturan di lembar kertas.

**Cara pakai**
- Jalankan sel dari atas ke bawah (**Shift + Enter**).
- Sel bertanda `# ISI` harus kamu lengkapi. Di tempat itu ada `...` atau nilai sementara supaya notebook tetap bisa dijalankan. Ganti dengan jawabanmu.
- Baris `cek(...)` memberi tanda ✅ atau ❌.
- **Bagian 1** dikerjakan bersama di kelas (live coding). **Bagian 2** adalah **tugas praktikmu**, dikumpulkan di akhir sesi.

## Bagian 0 · Satu alat bantu (jalankan saja)

`cek(nama, kondisi)` mencetak ✅ kalau `kondisi` benar, ❌ kalau salah. Hanya itu.

In [ ]:
def cek(nama, kondisi):
    if kondisi:
        print("✅ benar:", nama)
    else:
        print("❌ belum:", nama)

print("Alat siap.")

## Bagian 1 · Minimax dari kertas ke kode (dikerjakan bersama)

### 1.1 Pohon latihan sebagai data
Ini pohon yang sama dengan halaman 1 lembar kertas. `ANAK` menjawab "dari simpul ini, langkah apa saja yang mungkin?". `SKOR` berisi angka di daun: skor akhir permainan **untukmu** (makin besar makin baik buatmu).

In [ ]:
ANAK = {
    "akar": ["A", "B", "C"],       # giliranmu: tiga pilihan langkah
    "A": ["A1", "A2", "A3"],       # giliran lawan setelah kamu memilih A
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
}
SKOR = {"A1": 6, "A2": 8, "A3": 7,
        "B1": 9, "B2": 4, "B3": 7,
        "C1": 3, "C2": 5, "C3": 9}

def anak_pohon(s):
    """Daftar anak simpul s. Daun tidak punya anak."""
    if s in ANAK:
        return ANAK[s]
    return []

def skor_pohon(s):
    return SKOR[s]

print("Anak akar:", anak_pohon("akar"))
print("Anak A2  :", anak_pohon("A2"), "(daun)")
cek("B punya 3 anak", len(anak_pohon("B")) == 3)

### 1.2 Fungsi `minimax`: fungsi yang memanggil dirinya sendiri
Di kertas, untuk tahu nilai **akar** kamu perlu nilai **A, B, C** dulu. Untuk tahu nilai **A**, kamu perlu nilai **A1, A2, A3**. Itu persis yang dilakukan fungsi ini: ia memanggil `minimax` lagi untuk setiap anak, sampai bertemu daun.

- `giliran_max = True` artinya giliranmu (MAX). `False` artinya giliran lawan (MIN).
- `not giliran_max` membalik giliran: setelah kamu, giliran lawan, lalu kamu lagi.
- `dilihat` adalah list yang mencatat daun yang dilihat, supaya kita bisa menghitungnya.
- Kalau `cetak=True`, fungsi ini menuliskan setiap nilai yang dihitung, dengan menjorok sesuai kedalaman.

In [ ]:
def minimax(s, giliran_max, anak, skor, dilihat, kedalaman=0, cetak=False):
    jarak = "    " * kedalaman
    daftar_anak = anak(s)
    # Aturan 1: daun, nilainya sudah tertulis
    if len(daftar_anak) == 0:
        dilihat.append(s)
        if cetak:
            print(jarak + str(s), "= daun, nilai", skor(s))
        return skor(s)
    # Aturan 2: hitung nilai tiap anak dulu, dari kiri ke kanan
    nilai_anak = []
    for s2 in daftar_anak:
        v = minimax(s2, not giliran_max, anak, skor, dilihat, kedalaman + 1, cetak)
        nilai_anak.append(v)
    if giliran_max:
        # Aturan 3: giliranmu, ambil yang terbesar
        hasil = nilai_anak[-1]      # ISI: ganti (sementara: anak terakhir), pakai max(...)
    else:
        # Aturan 4: giliran lawan, ambil yang terkecil
        hasil = nilai_anak[-1]      # ISI: ganti (sementara: anak terakhir), pakai min(...)
    if cetak:
        if giliran_max:
            print(jarak + str(s), "= max dari", nilai_anak, "=", hasil)
        else:
            print(jarak + str(s), "= min dari", nilai_anak, "=", hasil)
    return hasil

dilihat = []
nilai_akar = minimax("akar", True, anak_pohon, skor_pohon, dilihat, cetak=True)
print()
print("Nilai akar:", nilai_akar, "| daun dilihat:", len(dilihat))
cek("nilai akar sama dengan kertas", nilai_akar == 6)

Cocokkan cetakan di atas dengan halaman 2 lembar kertasmu. Urutan barisnya sama dengan urutan kamu menghitung: anak-anak dulu, baru induknya.

**Pertanyaan 1.** Di cetakan, baris `A = min dari [...]` muncul **setelah** baris A1, A2, A3. Kenapa harus begitu?

> Jawabanmu: ...

### 1.3 Aturan 6: dari nilai ke keputusan
Nilai akar (6) belum menjawab pertanyaan kita. Yang kita butuhkan adalah **langkah mana** yang dipilih. Caranya: hitung nilai setiap anak akar, lalu ambil anak yang nilainya terbaik untuk yang sedang jalan.

In [ ]:
def pilih_langkah(s, giliran_max, anak, skor):
    """Aturan 6: kembalikan (anak terbaik, nilainya) untuk yang sedang jalan di s."""
    terbaik = None
    nilai_terbaik = None
    for s2 in anak(s):
        v = minimax(s2, not giliran_max, anak, skor, [])
        lebih_baik = False
        if terbaik is None:
            lebih_baik = True
        elif giliran_max and v > nilai_terbaik:
            lebih_baik = True
        elif not giliran_max and v < nilai_terbaik:
            lebih_baik = True
        if lebih_baik:
            terbaik = s2
            nilai_terbaik = v
    return terbaik, nilai_terbaik

for s2 in anak_pohon("akar"):
    print("Kalau kamu memilih", s2, "-> nilai", minimax(s2, False, anak_pohon, skor_pohon, []))

langkah, nilai = pilih_langkah("akar", True, anak_pohon, skor_pohon)
print("Pilih:", langkah, "dengan nilai", nilai)
cek("langkah yang dipilih", langkah == "A")

**Pertanyaan 2.** Daun terbesar di pohon ini adalah 9 (di B1 dan C3). Kenapa komputer tidak memilih B atau C untuk mengejar angka 9?

> Jawabanmu: langkah kita dan solusi terbaik kita akan diperngaruhi oleh langkah terbaik lawan dan dikta selalu beramsumsi bahwa lawan akan mengharappkan langkah terbutuk untuk kita


### 1.4 Aturan 5: memangkas cabang (alpha-beta)
Fungsi ini sama dengan `minimax`, ditambah dua catatan:

- `alfa` = nilai terbaik yang **sudah pasti** bisa kamu dapat (dari cabang yang sudah selesai dihitung).
- `beta` = nilai terburuk yang **sudah pasti** bisa dipaksakan lawan.

Kalau `alfa >= beta`, sisa anak di simpul ini tidak akan mengubah keputusan: berhenti (`break`). Di awal, `alfa = -1000` dan `beta = 1000`, artinya "belum tahu apa-apa".

In [ ]:
def alphabeta(s, giliran_max, anak, skor, alfa, beta, dilihat, kedalaman=0, cetak=False):
    jarak = "    " * kedalaman
    daftar_anak = anak(s)
    # Aturan 1: daun
    if len(daftar_anak) == 0:
        dilihat.append(s)
        if cetak:
            print(jarak + str(s), "= daun, nilai", skor(s))
        return skor(s)
    if giliran_max:
        terbaik = -1000
        for i in range(len(daftar_anak)):
            v = alphabeta(daftar_anak[i], False, anak, skor, alfa, beta, dilihat, kedalaman + 1, cetak)
            if v > terbaik:             # Aturan 3: simpan yang terbesar
                terbaik = v
            if terbaik > alfa:
                alfa = terbaik
            if alfa >= beta:            # Aturan 5: pangkas
                if cetak:
                    print(jarak + "  pangkas, tidak dilihat:", daftar_anak[i + 1:])
                break
    else:
        terbaik = 1000
        for i in range(len(daftar_anak)):
            v = alphabeta(daftar_anak[i], True, anak, skor, alfa, beta, dilihat, kedalaman + 1, cetak)
            if v < terbaik:             # Aturan 4: simpan yang terkecil
                terbaik = v
            if terbaik < beta:
                beta = terbaik
            if False:            # Aturan 5: pangkas  (ISI: ganti False dengan syarat pangkas)
                if cetak:
                    print(jarak + "  pangkas, tidak dilihat:", daftar_anak[i + 1:])
                break
    if cetak:
        print(jarak + str(s), "=", terbaik, "| alfa =", alfa, "| beta =", beta)
    return terbaik

dilihat_ab = []
nilai_ab = alphabeta("akar", True, anak_pohon, skor_pohon, -1000, 1000, dilihat_ab, cetak=True)
print()
print("Nilai akar:", nilai_ab, "| daun dilihat:", dilihat_ab)
cek("nilai sama dengan minimax", nilai_ab == 6)
cek("hanya 6 daun yang dilihat", len(dilihat_ab) == 6)

**Pertanyaan 3.** Di cabang B, setelah melihat B2 = 4, komputer berhenti. Tulis alasannya dengan kalimatmu sendiri, tanpa kata "alfa" dan "beta".

> Jawabanmu: ...

### 1.5 Tic-tac-toe
Papan ditulis sebagai teks 9 huruf, kotak 0 sampai 8 dari kiri atas ke kanan bawah. `.` = kosong.

```
0 1 2
3 4 5
6 7 8
```

Skor akhir: **1** kalau X menang, **-1** kalau O menang, **0** kalau seri. X adalah MAX, O adalah MIN.

In [ ]:
GARIS = [(0, 1, 2), (3, 4, 5), (6, 7, 8),      # mendatar
         (0, 3, 6), (1, 4, 7), (2, 5, 8),      # tegak
         (0, 4, 8), (2, 4, 6)]                 # miring

def tampilkan(papan):
    print(papan[0:3])
    print(papan[3:6])
    print(papan[6:9])

def pemenang(papan):
    for a, b, c in GARIS:
        if papan[a] != "." and papan[a] == papan[b] and papan[b] == papan[c]:
            return papan[a]
    return None

def giliran(papan):
    """X jalan duluan, jadi kalau jumlah X dan O sama, giliran X."""
    if papan.count("X") == papan.count("O"):
        return "X"
    return "O"

def anak_ttt(papan):
    """Semua papan setelah satu langkah. Kalau permainan selesai, tidak ada anak."""
    if pemenang(papan) is not None or "." not in papan:
        return []
    huruf = giliran(papan)
    hasil = []
    for i in range(9):
        if papan[i] == ".":
            baru = papan[:i] + huruf + papan[i + 1:]   # potong kiri + huruf baru + potong kanan
            hasil.append(baru)
    return hasil

def skor_ttt(papan):
    if pemenang(papan) == "X":
        return 1
    if pemenang(papan) == "O":
        return -1
    return 0

def kotak_diisi(lama, baru):
    """Nomor kotak yang berbeda antara dua papan."""
    for i in range(9):
        if lama[i] != baru[i]:
            return i

tampilkan("XO..X...O")
print("giliran:", giliran("XO..X...O"), "| banyak langkah:", len(anak_ttt("XO..X...O")))
cek("XXX di baris atas: X menang", pemenang("XXXOO....") == "X")

### 1.6 Teka-teki di awal kelas
Kamu X. Komputer mencoba setiap kotak kosong, menghitung nilainya dengan minimax, lalu memilih.

In [ ]:
teka = "OXOX....."
tampilkan(teka)
print()
for s2 in anak_ttt(teka):
    v = minimax(s2, False, anak_ttt, skor_ttt, [])
    print("X di kotak", kotak_diisi(teka, s2), "-> nilai", v)

langkah, nilai = pilih_langkah(teka, True, anak_ttt, skor_ttt)
print()
print("Pilih kotak", kotak_diisi(teka, langkah), "dengan nilai", nilai)
cek("kotak yang menjamin menang", kotak_diisi(teka, langkah) == 4 and nilai == 1)

### 1.7 Main melawan bot
Kamu X (jalan duluan), bot O. Bot memakai `pilih_langkah`, jadi ia selalu menganggap kamu bermain terbaik. Hapus tanda `#` di depan `main_lawan_bot()` di baris terakhir, jalankan sel, lalu ketik nomor kotak (0 sampai 8) dan tekan Enter. Setelah selesai bermain, beri lagi tanda `#` supaya Restart and run all tidak berhenti menunggu input.

In [ ]:
def main_lawan_bot():
    papan = "........."
    while len(anak_ttt(papan)) > 0:
        tampilkan(papan)
        if giliran(papan) == "X":
            kotak = int(input("Kotak untuk X (0-8): "))
            if kotak < 0 or kotak > 8 or papan[kotak] != ".":
                print("Kotak itu tidak bisa dipakai, coba lagi.")
                continue
            papan = papan[:kotak] + "X" + papan[kotak + 1:]
        else:
            langkah, nilai = pilih_langkah(papan, False, anak_ttt, skor_ttt)
            print("Bot memilih kotak", kotak_diisi(papan, langkah), "(nilai", nilai, ")")
            papan = langkah
        print()
    tampilkan(papan)
    print("Hasil:", pemenang(papan) if pemenang(papan) else "seri")

# main_lawan_bot()      # hapus tanda # di depan baris ini untuk bermain

Kalau `nilai` yang dicetak bot 0, artinya bot yakin hasilnya paling buruk seri. Kalau -1, bot yakin menang.

---

## Bagian 2 · TUGAS PRAKTIK

Kerjakan sendiri. Kalau macet, tanya dosen. **Dikumpulkan di akhir sesi.** Semua soal butuh `minimax`, `pilih_langkah`, dan `alphabeta` dari Bagian 1 yang sudah ✅.

### Soal 1 · Alpha-beta dengan tangan
Di **halaman 3 lembar kerja** ada pohon kedua. Jalankan minimax dan alpha-beta dengan tangan, coret daun yang tidak perlu dilihat, lalu tulis keputusanmu. **Baru** jalankan sel untuk mengecek.

In [ ]:
ANAK_2 = {"akar": ["P", "Q", "R"],
          "P": ["P1", "P2", "P3"], "Q": ["Q1", "Q2", "Q3"], "R": ["R1", "R2", "R3"]}
SKOR_2 = {"P1": 3, "P2": 9, "P3": 6, "Q1": 7, "Q2": 5, "Q3": 8, "R1": 4, "R2": 9, "R3": 2}

def anak_2(s):
    if s in ANAK_2:
        return ANAK_2[s]
    return []

def skor_2(s):
    return SKOR_2[s]

nilai_akar_2 = ...       # ISI: dari kertas
langkah_2 = ...          # ISI: "P", "Q", atau "R"
dicoret_2 = ["?"]          # ISI: daun yang dicoret, contoh ["P2", "P3"]

dilihat_2 = []
v2 = alphabeta("akar", True, anak_2, skor_2, -1000, 1000, dilihat_2)
pilihan_2, nilai_pilihan_2 = pilih_langkah("akar", True, anak_2, skor_2)
tidak_dilihat_2 = []
for daun in SKOR_2:
    if daun not in dilihat_2:
        tidak_dilihat_2.append(daun)
cek("nilai akar", nilai_akar_2 == v2)
cek("langkah yang dipilih", langkah_2 == pilihan_2)
cek("daun yang dicoret", sorted(dicoret_2) == sorted(tidak_dilihat_2))

Tulis satu kalimat: kenapa di cabang Q tidak ada daun yang dicoret, padahal di cabang R ada?

> Jawabanmu: ...

### Soal 2 · Urutan anak itu penting
Pohon latihan yang sama, tetapi langkah di akar diperiksa dengan urutan terbalik: C, B, lalu A.

In [ ]:
ANAK_BALIK = {"akar": ["C", "B", "A"],
              "A": ["A1", "A2", "A3"], "B": ["B1", "B2", "B3"], "C": ["C1", "C2", "C3"]}

def anak_balik(s):
    if s in ANAK_BALIK:
        return ANAK_BALIK[s]
    return []

dilihat_balik = []
v_balik = alphabeta("akar", True, anak_balik, skor_pohon, -1000, 1000, dilihat_balik, cetak=True)
print()
print("Nilai akar:", v_balik, "| daun dilihat:", len(dilihat_balik))

daun_dilihat_balik = 0     # ISI: dari cetakan
cek("nilai akar tetap 6", v_balik == 6)
cek("banyak daun yang dilihat", daun_dilihat_balik == len(dilihat_balik))

Tulis dua kalimat: nilai akarnya sama, tetapi banyak daun yang dilihat berbeda. Kenapa? Urutan seperti apa yang membuat alpha-beta memangkas paling banyak?

> Jawabanmu: ...

### Soal 3 · Teka-teki baru
Kamu X. Hitung nilai setiap kotak kosong dengan minimax.

In [ ]:
teka_3 = "OX.X..O.."
tampilkan(teka_3)
print()
for s2 in anak_ttt(teka_3):
    print("X di kotak", kotak_diisi(teka_3, s2), "-> nilai", minimax(s2, False, anak_ttt, skor_ttt, []))

kotak_menang_3 = -1       # ISI: kotak yang nilainya 1
kotak_seri_3 = -1         # ISI: kotak yang nilainya 0
kotak_kalah_3 = []        # ISI: list kotak yang nilainya -1, contoh [2, 5]

nilai_3 = {}
for s2 in anak_ttt(teka_3):
    nilai_3[kotak_diisi(teka_3, s2)] = minimax(s2, False, anak_ttt, skor_ttt, [])
cek("kotak menang", nilai_3.get(kotak_menang_3) == 1)
cek("kotak seri", nilai_3.get(kotak_seri_3) == 0)
semua_kalah = []
for k in nilai_3:
    if nilai_3[k] == -1:
        semua_kalah.append(k)
cek("kotak kalah", sorted(kotak_kalah_3) == sorted(semua_kalah))

Tulis satu kalimat: kenapa kotak yang menang itu menjamin kemenangan, apa pun yang dilakukan O? (Petunjuk: hitung berapa garis yang terancam X sesudahnya.)

> Jawabanmu: ...

### Soal 4 · Berapa yang dipangkas?
Hitung banyak posisi akhir (daun) yang dilihat dari papan **kosong**, dengan minimax dan dengan alpha-beta. Sel ini butuh beberapa detik.

In [ ]:
dilihat_mm = []
v_mm = minimax(".........", True, anak_ttt, skor_ttt, dilihat_mm)
dilihat_ab_ttt = []
v_ab = alphabeta(".........", True, anak_ttt, skor_ttt, -1000, 1000, dilihat_ab_ttt)
print("Minimax   | nilai:", v_mm, "| daun dilihat:", len(dilihat_mm))
print("Alpha-beta| nilai:", v_ab, "| daun dilihat:", len(dilihat_ab_ttt))

daun_minimax = 0         # ISI: dari cetakan
daun_alphabeta = 0       # ISI: dari cetakan
persen = 0               # ISI: daun_alphabeta dibagi daun_minimax, dikali 100
cek("daun minimax", daun_minimax == len(dilihat_mm))
cek("daun alpha-beta", daun_alphabeta == len(dilihat_ab_ttt))
cek("persen", abs(persen - 100 * len(dilihat_ab_ttt) / len(dilihat_mm)) < 0.1)

Tulis dua kalimat: (1) nilai papan kosong adalah 0. Apa artinya untuk permainan tic-tac-toe? (2) Apakah alpha-beta pernah memberi nilai yang berbeda dari minimax?

> Jawabanmu: ...

### Soal 5 · Kalau tidak sanggup melihat sampai akhir
Di catur, pohonnya terlalu besar untuk dihitung sampai akhir. Program catur berhenti di **batas kedalaman**, lalu menebak nilai posisi dengan **fungsi evaluasi** buatan manusia.

Fungsi evaluasi kita: untuk setiap garis, **+1** kalau garis itu belum ada O (X masih bisa menang di sana), **-1** kalau belum ada X. Posisi akhir yang sebenarnya diberi nilai **100** atau **-100**, supaya kemenangan pasti selalu lebih berharga dari tebakan.

In [ ]:
def evaluasi(papan):
    total = 0
    for a, b, c in GARIS:
        isi = [papan[a], papan[b], papan[c]]
        if "O" not in isi:
            total = total + 1
        if "X" not in isi:
            total = total             # ISI
    return total

def minimax_terbatas(s, giliran_max, batas):
    daftar_anak = anak_ttt(s)
    if len(daftar_anak) == 0:
        return 100 * skor_ttt(s)       # posisi akhir yang sebenarnya
    if batas == 0:
        return evaluasi(s)             # berhenti melihat, tebak nilainya
    nilai_anak = []
    for s2 in daftar_anak:
        nilai_anak.append(minimax_terbatas(s2, not giliran_max, batas - 1))
    if giliran_max:
        return max(nilai_anak)
    return min(nilai_anak)

def pilih_terbatas(s, batas):
    """Pilihan X kalau hanya sanggup melihat `batas` langkah setelah langkahnya sendiri."""
    terbaik = None
    nilai_terbaik = None
    for s2 in anak_ttt(s):
        v = minimax_terbatas(s2, False, batas)
        if terbaik is None or v > nilai_terbaik:
            terbaik = s2
            nilai_terbaik = v
    return kotak_diisi(s, terbaik)

cek("papan kosong: evaluasi 0", evaluasi(".........") == 0)
cek("X di tengah: evaluasi 4", evaluasi("....X....") == 4)
for batas in [0, 1, 2, 3]:
    print("batas", batas, "-> X memilih kotak", pilih_terbatas(teka_3, batas))

pilihan_batas_1 = -1           # ISI: kotak yang dipilih dengan batas 1
batas_terkecil_benar = -1      # ISI: batas terkecil yang memilih kotak menang dari Soal 3
cek("pilihan dengan batas 1", pilihan_batas_1 == pilih_terbatas(teka_3, 1))
cek("batas terkecil yang benar", batas_terkecil_benar == 2)

Tulis dua kalimat: dengan batas 1, X memilih kotak yang tidak terbaik. Kenapa fungsi evaluasi bisa salah menilai? Apa untungnya batas yang lebih kecil?

> Jawabanmu: ...

### Soal 6 · Refleksi
Dua pertanyaan, masing-masing satu atau dua kalimat:

1. Deep Blue (1997) mengalahkan juara dunia catur dengan alpha-beta dan fungsi evaluasi yang **ditulis manusia**. Kenapa fungsi evaluasi untuk catur sulit ditulis dengan tangan?
2. Minimax menganggap lawan selalu bermain terbaik. Beri satu situasi (game atau di luar game) di mana anggapan itu kurang tepat.

> Jawabanmu 1: ...
>
> Jawabanmu 2: ...

---
**Sebelum dikumpulkan:** pilih **Runtime → Restart and run all** dan pastikan tidak ada sel yang error. Pastikan baris `main_lawan_bot()` di sel 1.7 diberi tanda `#` lagi. Simpan notebook (File → Download .ipynb) lalu kumpulkan.